Кольчурин Андрей Николавеич

студент физико-математического института,

направления радиофизика,

группы ФМ/О РФЗ-1-2022 НБ

In [2]:
%%capture
!pip install jaxlib
!pip install jax
!pip install git+https://github.com/deepmind/dm-haiku
!pip install gym==0.25
!pip install gym[box2d]
!pip install optax
!pip install matplotlib
!pip install chex
!pip install gym[classic_control]

In [3]:
import copy
from shutil import rmtree # deleting directories
import random
import collections # useful data structures
import numpy as np
import gym # reinforcement learning environments
from gym.wrappers import RecordVideo
import jax
import jax.numpy as jnp # jax numpy
import haiku as hk # jax neural network library
import optax # jax optimizer library
import matplotlib.pyplot as plt # graph plotting library
from IPython.display import HTML
from base64 import b64encode
import chex

# Hide warnings
import warnings
warnings.filterwarnings('ignore')

In [6]:
# Create the environment
env_name = "CartPole-v0"
env = gym.make(env_name)

In [7]:
# Reset the environment
s_0 = env.reset()
print("Initial State::", s_0)

# Get environment obs space
obs_shape = env.observation_space.shape
print("Environment Obs Space Shape:", obs_shape)

Initial State:: [ 0.00188594 -0.03512136 -0.02118021  0.03364546]
Environment Obs Space Shape: (4,)


In [8]:
# Get action space - e.g. discrete or continuous
print(f"Environment action space: {env.action_space}")

# Get num actions
num_actions = env.action_space.n
print(f"Number of actions: {num_actions}")

Environment action space: Discrete(2)
Number of actions: 2


# Упражнение 1

In [15]:
def linear_policy(params, obs):
  """A simple linear policy"""
  # YOUR CODE
  dot_product_result = jnp.dot(params, obs)

  action = jax.lax.select(
      dot_product_result > 0,  # boolean statement goes here
      1,                       # result when the statement is True
      0                        # result when the statement is False
  )
  # END YOUR CODE
  return action


Your function is correct! 


ПРоверка

In [14]:
def check_linear_policy(linear_policy):
  fixed_obs = jnp.array([1,1,2,4])
  params1 = jnp.array([1,1,1,1])
  params2 = jnp.array([-1,-1,-1,-1])
  hint1 = f"Неправильный ответ, ваша линейная политика неверна. Действие при \n obs={fixed_obs} и params={params1} должно быть 1 "
  hint2 = f"Неправильный ответ, ваша линейная политика неверна. Действие при \n obs={fixed_obs} и params={params2} должно быть 0 "
  hint = None
  if linear_policy(params1, fixed_obs) != 1: hint = hint1
  elif linear_policy(params2, fixed_obs) != 0: hint = hint2
  if hint is not None: print(hint)
  else: print("Your function is correct! ")

try: check_linear_policy(linear_policy)
except Exception as e: print("An Error Occured: {} ".format(e))

Your function is correct! 


# Упражнение 2

In [28]:
def run_episode(env):
  episode_return = 0
  done = False
  params = jnp.array([1,-2,2,-1])

  ## YOUR CODE
  obs = env.reset() # TODO: get the initial obs from the env

  while not done:
    action = linear_policy(params, obs) # TODO: compute action using linear policy
    action = np.array(action)
    obs, reward, done, info = env.step(action) # TODO: step the environment
    episode_return += reward # TODO: add reward to episode return
  # END YOUR CODE
  return episode_return

Проверка

In [29]:

try:
  env.seed(42)
  if run_episode(env) == 31: print("Looks correct! ")
  else: print("Looks like your implementation might be wrong. ")
except Exception as e: print("An Error Occured: {} ".format(e))

Looks correct! 


# Упражнение 3

In [34]:
# Parameter container for Random Policy Search
RandomPolicySearchParams = collections.namedtuple("RandomPolicySearchParams", ["current", "best"])

# TEST: store two different sets of parameters
current_params = np.ones(obs_shape) * -1
best_params = np.zeros(obs_shape)
rps_params = RandomPolicySearchParams(current_params, best_params)

# How to access the best or current params.
print(f"Best params: {rps_params.best}")
print(f"Current params: {rps_params.current}")

Best params: [0. 0. 0. 0.]
Current params: [-1. -1. -1. -1.]


In [35]:
def random_policy_search_choose_action(key, params, actor_state, obs, evaluation=False):
  # YOUR CODE
  best_action = linear_policy(params.best, obs)
  current_action = linear_policy(params.current, obs)
  action = jax.lax.select(evaluation, best_action, current_action)
  # END YOUR CODE
  return action, actor_state

Проверка


In [36]:
def check_random_policy_search_choose_action(choose_action):
  key = None; actor_state = None; obs = np.ones(obs_shape)
  evaluation=False; rps_params = RandomPolicySearchParams(np.ones(obs_shape)*-1, np.ones(obs_shape))
  action, _ = choose_action(key,rps_params,actor_state,obs,evaluation)
  if action != 0: return False
  evaluation=True; rps_params = RandomPolicySearchParams(np.ones(obs_shape)*-1, np.ones(obs_shape))
  action, _ = choose_action(key,rps_params,actor_state,obs,evaluation)
  if action != 1: return False
  return True
try:
  if check_random_policy_search_choose_action(random_policy_search_choose_action): print("Your function looks correct. ")
  else: print("Your function looks incorrect. ")
except Exception as e: print("An Error Occured: {} ".format(e))

Your function looks correct. 


# Упражнение 4

In [93]:
def get_new_random_weights(random_key, old_weights, minval=-2.0, maxval=2.0):
    new_weights_shape = old_weights.shape
    new_weights_dtype = old_weights.dtype
    # YOUR CODE
    new_params = jax.random.uniform(random_key, shape=new_weights_shape, minval=minval, maxval=maxval, dtype=new_weights_dtype)
    # END YOUR CODE
    return new_params

Проверка

In [94]:
def check_get_new_random_weights(get_new_random_weights):
  old_weights = np.ones(obs_shape,  "float32")
  random_key = jax.random.PRNGKey(42)
  new_weights = get_new_random_weights(random_key, old_weights, minval=-2.0, maxval=2.0)
  if jnp.array_equal(new_weights, jnp.array([ 0.29657745,1.4265499, -1.7621555, -1.7505779 ])): print("Function is correct! ")
  else: print("Something is wrong. ")
try: check_get_new_random_weights(get_new_random_weights)
except Exception as e: print("An Error Occured: {} ".format(e))

Something is wrong. 


# Упражнение 5

In [48]:
# A NamedTuple to store the best average episode return so far
RandomPolicyLearnState = collections.namedtuple(
  "RandomPolicyLearnState",
  ["best_average_episode_return"]
)

# Test
initial_learn_state = RandomPolicyLearnState(best_average_episode_return=-float("inf"))
print("Initial best average episode return:", initial_learn_state.best_average_episode_return)

Initial best average episode return: -inf


In [49]:
def random_policy_search_learn(key, params, learn_state, memory):
    best_params = params.best
    current_params = params.current
    current_average_episode_return = memory
    best_average_episode_return = learn_state.best_average_episode_return

    # YOUR CODE
    best_params = jax.lax.select(
        current_average_episode_return > best_average_episode_return,
        current_params,
        best_params
    )

    best_average_episode_return = jax.lax.select(
        current_average_episode_return > best_average_episode_return,
        current_average_episode_return,
        best_average_episode_return
    )
    # END YOUR CODE

    new_params = get_new_random_weights(key, current_params)
    params = RandomPolicySearchParams(current=new_params, best=best_params)

    return params, RandomPolicyLearnState(best_average_episode_return)

In [50]:
params = RandomPolicySearchParams(np.ones(obs_shape, "float32"), np.ones(obs_shape, "float32") * -1)
learn_state = RandomPolicyLearnState(10); memory = 11; key = jax.random.PRNGKey(42)
try:
  new_params, new_learn_state = random_policy_search_learn(key, params, learn_state, memory)
  if not jnp.array_equal(new_params.current, jnp.array([ 0.29657745,  1.4265499 , -1.7621555 , -1.7505779 ])): print("Your function is incorrect. ")
  elif not jnp.array_equal(new_params.best, jnp.array([1., 1., 1., 1.])): print("Your function is incorrect. ")
  elif new_learn_state.best_average_episode_return != 11: print("Your function is incorrect. ")
  else: print("Your function looks correct. ")
except Exception as e: print("An Error Occured: {} ".format(e))

Your function is incorrect. 


# Упражнение 6

In [46]:
def compute_weighted_log_prob(action_prob, episode_return):
    # YOUR CODE
    log_prob = jnp.log(action_prob)
    weighted_log_prob = log_prob * episode_return
    # END YOUR CODE
    return weighted_log_prob

Проверка

In [47]:

try:
  action_prob = 0.8; episode_return = 100
  result = compute_weighted_log_prob(action_prob, episode_return)
  if result != -22.314354: print("Your implementation looks incorrect. ")
  else: print("Looks correct. ")
except Exception as e: print("An Error Occured: {} ".format(e))

Looks correct. 


# Упражнение 7

In [51]:
def compute_rewards_to_go(rewards):
    rewards_to_go = []
    # YOUR CODE
    cumulative = 0
    for r in reversed(rewards):
        cumulative += r
        rewards_to_go.append(cumulative)
    rewards_to_go = list(reversed(rewards_to_go))
    # END YOUR CODE
    return rewards_to_go

Проверка


In [52]:

try:
  result = compute_rewards_to_go([1,2,3,4])
  if result != [10, 9, 7, 4]: print("There is a problem with your implementation. ")
  else: print("Looks correct. ")
except Exception as e: print("An Error Occured: {} ".format(e))

Looks correct. 


# Упражнение 8

In [53]:
def sample_action(random_key, logits):
  # YOUR CODE HERE
  action = jax.random.categorical(random_key, logits)
  # END YOUR code
  return action

Проверка

In [55]:

try:
  random_key = jax.random.PRNGKey(42)
  action = sample_action(random_key, np.array([1,2],  "float32"))
  if action != 1: print("Your function is incorrect. ")
  else: print("Seems correct. ")
except Exception as e: print("An Error Occured: {} ".format(e))

Seems correct. 


# Упражнение 9

In [96]:
def policy_gradient_loss(action, logits, reward_to_go):
  # YOUR CODE
  all_action_probs = jax.nn.softmax(logits)
  action_prob = all_action_probs[action]
  weighted_log_prob = compute_weighted_log_prob(action_prob, reward_to_go)
  # END YOUR CODE
  loss = - weighted_log_prob
  return loss

Проверка

In [97]:

try:
  result = policy_gradient_loss(1, np.array([1,2],  "float32"), 10)
  if result != 3.1326175: print("Your implementation looks wrong. ")
  else: print("Looks correct. ")
except Exception as e: print("An Error Occured: {} ".format(e))

Your implementation looks wrong. 


# Упражнение 10

In [59]:
def select_greedy_action(q_values):
  # YOUR CODE
  action = jnp.argmax(q_values)
  # END YOUR CODE
  return action

проверка


In [60]:

try:
  q_values = jnp.array([1,1,3,4])
  action = select_greedy_action(q_values)
  if action != 3: print("Incorrect answer, your greedy action selector looks wrong ")
  else: print("Looks good. ")
except Exception as e: print("An Error Occured: {} ".format(e))

Looks good. 


# Упражнение 11

In [61]:
def compute_squared_error(pred, target):
  # YOUR CODE
  squared_error = jnp.square(pred - target)
  # END YOUR CODE
  return squared_error

Проверка


In [62]:

try:
  result = compute_squared_error(1, 4)
  if result != 9: print("Your implementation looks wrong.")
  else: print("Looks good.")
except Exception as e: print("An Error Occured: {} ".format(e))

Looks good.


# Упражнение 12

In [63]:
def compute_bellman_target(reward, done, next_q_values):
  # YOUR CODE
  bellman_target = reward + (1.0 - done) * jnp.max(next_q_values)
  # END YOUR CODE
  return bellman_target

Проверка

In [65]:

try:
  result1 = compute_bellman_target(1, 0.0, np.array([3,2],  "float32"))
  result2 = compute_bellman_target(1, 1.0, np.array([3,2],  "float32"))
  if result1 != 4 or result2 != 1: print("Your implementation looks wrong.")
  else: print("Looks good.")
except Exception as e: print("An Error Occured: {} ".format(e))

Looks good.


# Упражнение 13

In [66]:
def q_learning_loss(q_values, action, reward, done, next_q_values):
  # YOUR CODE
  chosen_action_q_value = q_values[action]
  bellman_target = compute_bellman_target(reward, done, next_q_values)
  squared_error = compute_squared_error(chosen_action_q_value, bellman_target)
  # END YOUR CODE
  return squared_error

Проверка


In [68]:

try:
  result = q_learning_loss(np.array([3,2],  "float32"), 1, 2, 0.0, np.array([3,2],  "float32"))
  if result != 9.0: print("Your implementation looks wrong.")
  else: print("Looks good.")
except Exception as e: print("An Error Occured: {} ".format(e))

Looks good.


# Упражнение 14

In [106]:
def select_random_action(key, num_actions):
    action = jax.random.randint(key, shape=(), minval=0, maxval=num_actions, dtype=jnp.int32)
    return action

Проверка

In [107]:
try:
  random_key1 = jax.random.PRNGKey(6)
  random_key2 = jax.random.PRNGKey(1000)
  result1 = select_random_action(random_key1, 2)
  result2 = select_random_action(random_key2, 2)
  if result1 != 1 or result2 != 0: print("Your implementation looks wrong.")
  else: print("Looks good.")
except: print("Your implementation looks wrong.")

Your implementation looks wrong.


# Упражнение 15

In [108]:
EPSILON_DECAY_TIMESTEPS = 3000 # decay epsilon over 3000 timesteps
EPSILON_MIN = 0.1 # 10% exploration

In [109]:
def get_epsilon(num_timesteps):
  # YOUR CODE
  epsilon = 1.0 - (num_timesteps / EPSILON_DECAY_TIMESTEPS)
  epsilon = jax.lax.select(
      epsilon < EPSILON_MIN,
      EPSILON_MIN,
      epsilon
  )
  # END YOUR CODE
  return epsilon

Проверка

In [110]:

def check_get_epsilon(get_epsilon):
  try:
    result1 = get_epsilon(10); result2 = get_epsilon(5_010)
    if result1 != 0.99666667 or result2 != 0.1: print("Your function looks wrong.")
    else: print("Your function looks correct.")
  except: print("Your function looks wrong.")
check_get_epsilon(get_epsilon)

Your function looks correct.


# Упражнение 16

In [85]:
def select_epsilon_greedy_action(key, q_values, num_timesteps):
    num_actions = len(q_values)
    epsilon = get_epsilon(num_timesteps)

    key_explore, key_action = jax.random.split(key)
    should_explore = jax.random.uniform(key_explore, shape=()) < epsilon

    action = jax.lax.cond(
        should_explore,
        lambda: select_random_action(key_action, num_actions),
        lambda: select_greedy_action(q_values)
    )

    return action

Проверка

In [86]:

try:
  rng = hk.PRNGSequence(jax.random.PRNGKey(42))
  dummy_q_values = jnp.array([0,1], jnp.float32)
  num_timesteps = 5010
  actions1 = [int(select_epsilon_greedy_action(next(rng), dummy_q_values, num_timesteps)) for _ in range(10)]
  num_timesteps = 0
  actions2 = [int(select_epsilon_greedy_action(next(rng), dummy_q_values, num_timesteps)) for _ in range(10)]
  if actions1 != [1, 1, 0, 1, 1, 0, 1, 1, 1, 1] or actions2 != [0, 0, 0, 1, 1, 1, 1, 0, 0, 0]:
    print("Looks like something might be incorrect!")
  else: print("Looks correct!")
except: print("Looks like something might be incorrect!")

Looks like something might be incorrect!
